In [1]:
!pip install google-play-scraper langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 38.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=264026d6abe4af67b4cd720b9c5b4a7db8e2ed4ae459f71f4fbe14c3f2ba0f14
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [2]:
from google_play_scraper import reviews, Sort
import pandas as pd

all_reviews = []
continuation_token = None

for _ in range(250):  # 250 * 200 ≈ 50K
    result, continuation_token = reviews(
        'com.openai.chatgpt',
        lang='en',
        country='us',
        sort=Sort.NEWEST,
        count=200,
        continuation_token=continuation_token
    )

    all_reviews.extend(result)

    if not continuation_token:
        break

df = pd.DataFrame(all_reviews)

In [3]:
df.to_csv('chatgpt_reviews_raw.csv', index=False)

In [4]:
df = df[['content', 'score']]

# убираем пустые
df = df[df['content'].notna()]

# убираем дубликаты
df = df.drop_duplicates()

# убираем короткие
df = df[df['content'].str.len() > 20]

df = df.reset_index(drop=True)

In [5]:
from langdetect import detect

def is_english(text):
    try:
        return detect(text) == 'en'
    except:
        return False

df = df[df['content'].apply(is_english)]
df = df.reset_index(drop=True)

In [6]:
print(df.shape)

(13611, 2)


In [7]:
negative = df[df['score'].isin([1, 2])]
positive = df[df['score'].isin([4, 5])]

print("Negative:", len(negative))
print("Positive:", len(positive))

Negative: 2310
Positive: 10564


In [8]:
n = min(len(negative), len(positive))

negative_sample = negative.sample(n, random_state=42)
positive_sample = positive.sample(n, random_state=42)

final_df = pd.concat([negative_sample, positive_sample])
final_df = final_df.reset_index(drop=True)

In [9]:
final_df.to_csv('chatgpt_reviews_clean_balanced.csv', index=False)

In [10]:
from google.colab import files

files.download('chatgpt_reviews_raw.csv')
files.download('chatgpt_reviews_clean_balanced.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
df['score'].value_counts()

,count
score,
5,9067
1,1864
4,1497
3,737
2,446
